In [5]:
import mlflow
print(mlflow.__version__)

3.14.0


In [6]:
from Cnnclassifier.constants import *
from Cnnclassifier.utils.common import read_yaml, create_directories, save_json

In [7]:
import os

print(os.listdir(r"E:\ai\deep learning project\src\Cnnclassifier\components"))

['data_ingestion.py', 'model_evaluation_mlflow.py', 'model_training.py', 'prepare_base_model.py', '__init__.py', '__pycache__']


In [8]:
import inspect
from Cnnclassifier.components.model_evaluation_mlflow import Evaluation

print(inspect.getsource(Evaluation.log_into_mlflow))

    def log_into_mlflow(self):

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)

        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        print("Tracking URI :", mlflow.get_tracking_uri())
        print("Registry URI :", mlflow.get_registry_uri())

        with mlflow.start_run():

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })

            if tracking_url_type_store != "file":

                mlflow.keras.log_model(
                    self.model,
                    "model",
                    registered_model_name="VGG16Model"
                )

            else:
                mlflow.keras.log_model(
                self.model,
                "model"
            )
                # Register the model
                # There are other ways to use the Mo

In [9]:
import os
os.chdir("../")

In [10]:
%pwd

'e:\\ai\\deep learning project'

In [11]:
import os

os.chdir(r"E:\ai\deep learning project")

print("Current Directory:", os.getcwd())

Current Directory: E:\ai\deep learning project


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [13]:
import tensorflow as tf

In [14]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

[2026-08-03 21:04:06,401: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]


In [15]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [16]:
from Cnnclassifier.constants import *
from Cnnclassifier.utils.common import read_yaml, create_directories, save_json

In [17]:
from Cnnclassifier.entity.config_entity import EvaluationConfig

print(EvaluationConfig)

<class 'Cnnclassifier.entity.config_entity.EvaluationConfig'>


In [ ]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"),
            mlflow_uri="https://dagshub.com/bogiereddy/kidney--disease-prediction.mlflow",
            all_params=self.params,
            params_image_size=self.params["IMAGE_SIZE"],
            params_batch_size=self.params["BATCH_SIZE"]
        )
        return eval_config

In [19]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [20]:
import mlflow.keras
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        print("===== MLFLOW STARTED =====")
        with mlflow.start_run():
             
             mlflow.log_params(self.config.all_params)

             mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })

             mlflow.keras.log_model(
                model=self.model,
                name="model",
                registered_model_name="VGG16Model"
            )
             print("model register")
    
    
            # Model registry does not work with file store
            

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                


In [21]:
import os

print("URI:", os.getenv("MLFLOW_TRACKING_URI"))
print("USERNAME:", os.getenv("MLFLOW_TRACKING_USERNAME"))
print("PASSWORD:", os.getenv("MLFLOW_TRACKING_PASSWORD"))

URI: https://dagshub.com/bogiereddy/kidney--disease-prediction.mlflow
USERNAME: bogiereddy
PASSWORD: dee8b528330ba44c9e45edb01be1684f171a697b


In [22]:
import os

print(os.getenv("MLFLOW_TRACKING_URI"))

https://dagshub.com/bogiereddy/kidney--disease-prediction.mlflow


In [23]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-08-03 21:04:31,685: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-03 21:04:31,689: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-03 21:04:31,690: INFO: common: created directory at: artifacts]
[2026-08-03 21:04:31,862: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 2207 images belonging to 2 classes.
138/138 ━━━━━━━━━━━━━━━━━━━━ 238s 2s/step - accuracy: 0.8999 - loss: 0.2781
[2026-08-03 21:08:30,422: INFO: common: json file saved at: scores.json]
===== MLFLOW STARTED =====


2026/08/03 21:08:36 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/08/03 21:09:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


model register
🏃 View run sedate-cow-213 at: https://dagshub.com/bogiereddy/kidney--disease-prediction.mlflow/#/experiments/0/runs/11bb51b3674a4a98adcec646e448ef3d
🧪 View experiment at: https://dagshub.com/bogiereddy/kidney--disease-prediction.mlflow/#/experiments/0
